In [ ]:

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os

print("TensorFlow version:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices("GPU")) > 0)

# Check available hardware
print("Devices:", tf.config.list_physical_devices())


TensorFlow version: 2.21.0
GPU available: True
Devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:

import os

drive_path = "/content/drive/MyDrive"

for root, dirs, files in os.walk(drive_path):
    dirs[:] = [d for d in dirs if not d.startswith(".")]

    if files:
        print(f"\nFolder: {root}")
        print("Files:", files[:5])



Folder: /content/drive/MyDrive
Files: ['my photo.mp4', 'मानक क्रमांक-11.pdf', 'मानक क्रमांक-46.pdf', 'मानक क्रमांक-123.pdf', 'मानक क्रमांक-117 (1).pdf']

Folder: /content/drive/MyDrive/Colab Notebooks
Files: ['BoviCare_Model_Training.ipynb']

Folder: /content/drive/MyDrive/appsheet/data/केंद्रभिनारबदलीशिक्षकमाहिती-3673908
Files: ['empty.txt']

Folder: /content/drive/MyDrive/Seva pustak
Files: ['seva pustak .pdf']

Folder: /content/drive/MyDrive/A SQAAF शाळा धामणगाव
Files: ['मानक क्रमांक-36.pdf', 'मानक क्रमांक-35.pdf', 'मानक क्रमांक-104.pdf', 'मानक क्रमांक-125.pdf', 'मानक क्रमांक-38.pdf']

Folder: /content/drive/MyDrive/Google Earth
Files: ['Hlb7', 'Untitled map (1)', 'Untitled map']


In [ ]:

import os

drive_path = "/content/drive/MyDrive"

for root, dirs, files in os.walk(drive_path):
    dirs[:] = [d for d in dirs if not d.startswith(".")]

    for file in files:
        if file.lower().endswith((".zip", ".jpg", ".jpeg", ".png")):
            print(os.path.join(root, file))


/content/drive/MyDrive/IMG_20250815_181545.jpg
/content/drive/MyDrive/IMG_20250815_232124.jpg


In [ ]:

import os

drive_path = "/content/drive/MyDrive"

for item in os.listdir(drive_path):
    full_path = os.path.join(drive_path, item)

    if os.path.isdir(full_path):
        print("FOLDER:", item)
    elif item.lower().endswith(".zip"):
        print("ZIP FILE:", item)


FOLDER: Colab Notebooks
FOLDER: appsheet
FOLDER: Seva pustak
FOLDER: Sqaff 2026
FOLDER: A SQAAF शाळा धामणगाव
FOLDER: Google Earth
FOLDER: Classroom


In [ ]:

import os
import zipfile

zip_path = "/content/drive/MyDrive/BoviCare_Dataset/archive (2).zip"
extract_path = "/content/bovicare_dataset"

# Check that the ZIP file exists
if not os.path.exists(zip_path):
    raise FileNotFoundError(f"Dataset ZIP not found: {zip_path}")

# Extract the dataset
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")
print("Extraction location:", extract_path)

# Show the folder structure
for root, dirs, files in os.walk(extract_path):
    dirs[:] = [d for d in dirs if not d.startswith(".")]
    image_files = [
        f for f in files
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
    ]

    if image_files:
        print(f"\nFolder: {root}")
        print("Number of images:", len(image_files))


Dataset extracted successfully!
Extraction location: /content/bovicare_dataset


In [ ]:

import os

root_path = "/content/bovicare_dataset"

print("Top-level contents:")
print(os.listdir(root_path))

print("\nAll folders inside the dataset:")
for root, dirs, files in os.walk(root_path):
    level = root.replace(root_path, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files[:3]:
        print(f"{indent}  {file}")


Top-level contents:
['cow_milk_mastitis_dataset.csv']

All folders inside the dataset:
bovicare_dataset/
  cow_milk_mastitis_dataset.csv


In [ ]:

import os
import zipfile

zip_path = "/content/drive/MyDrive/BoviCare_Dataset/archive (2) (1).zip"
extract_path = "/content/bovicare_images"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Image dataset extracted!")

for root, dirs, files in os.walk(extract_path):
    image_files = [
        f for f in files
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
    ]

    if image_files:
        print(f"\nFolder: {root}")
        print("Image count:", len(image_files))


Image dataset extracted!

Folder: /content/bovicare_images/Mastitis/Mastitis
Image count: 302

Folder: /content/bovicare_images/Mastitis/Normal
Image count: 299


In [ ]:

import os
import random
import shutil
from pathlib import Path

random.seed(42)

source_dirs = {
    "Mastitis": "/content/bovicare_images/Mastitis/Mastitis",
    "Normal": "/content/bovicare_images/Mastitis/Normal",
}

output_dir = Path("/content/bovicare_split")
image_extensions = {".jpg", ".jpeg", ".png", ".bmp"}

# Collect and validate image files
class_images = {}

for class_name, folder in source_dirs.items():
    folder_path = Path(folder)

    if not folder_path.exists():
        raise FileNotFoundError(f"Folder not found: {folder}")

    images = [
        p for p in folder_path.iterdir()
        if p.is_file() and p.suffix.lower() in image_extensions
    ]

    if len(images) < 3:
        raise ValueError(f"Too few images in {class_name}: {len(images)}")

    class_images[class_name] = images

# Create a fresh output directory
if output_dir.exists():
    shutil.rmtree(output_dir)

for split in ["train", "val", "test"]:
    for class_name in source_dirs:
        (output_dir / split / class_name).mkdir(
            parents=True, exist_ok=True
        )

# Split each class independently
for class_name, images in class_images.items():
    random.shuffle(images)

    total = len(images)
    train_end = int(total * 0.70)
    val_end = train_end + int(total * 0.15)

    splits = {
        "train": images[:train_end],
        "val": images[train_end:val_end],
        "test": images[val_end:],
    }

    for split_name, split_images in splits.items():
        for index, image_path in enumerate(split_images):
            destination = (
                output_dir / split_name / class_name /
                f"{class_name}_{index:04d}{image_path.suffix.lower()}"
            )
            shutil.copy2(image_path, destination)

# Display counts
for split in ["train", "val", "test"]:
    print(f"\n{split.upper()}")
    for class_name in source_dirs:
        folder = output_dir / split / class_name
        count = sum(1 for p in folder.iterdir() if p.is_file())
        print(f"{class_name}: {count}")

print("\nDataset split created at:", output_dir)



TRAIN
Mastitis: 211
Normal: 209

VAL
Mastitis: 45
Normal: 44

TEST
Mastitis: 46
Normal: 46

Dataset split created at: /content/bovicare_split


In [ ]:

import tensorflow as tf

# Image settings
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
SEED = 42

TRAIN_DIR = "/content/bovicare_split/train"
VAL_DIR = "/content/bovicare_split/val"
TEST_DIR = "/content/bovicare_split/test"

# Load training images
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True,
    seed=SEED
)

# Load validation images
val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

# Load test images
test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

# Confirm class order
print("Class names:", train_ds.class_names)

# Improve input pipeline performance
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

print("\nImages loaded successfully!")


Found 420 files belonging to 2 classes.
Found 89 files belonging to 2 classes.
Found 92 files belonging to 2 classes.
Class names: ['Mastitis', 'Normal']

Images loaded successfully!


In [ ]:

import tensorflow as tf
from tensorflow.keras import layers, models

# Use a pretrained MobileNetV2 model
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze pretrained layers for the first training stage
base_model.trainable = False

# Build the BoviCare classifier
inputs = layers.Input(shape=(224, 224, 3))

# MobileNetV2 preprocessing expects pixel values scaled to [-1, 1]
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = models.Model(inputs, outputs)

# Configure model training
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

# Display the architecture
model.summary()


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stopping]
)


Epoch 1/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 25s 491ms/step - accuracy: 0.6381 - loss: 0.6614 - precision: 0.6492 - recall: 0.5933 - val_accuracy: 0.7640 - val_loss: 0.4545 - val_precision: 0.7170 - val_recall: 0.8636
Epoch 2/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 2s 55ms/step - accuracy: 0.8214 - loss: 0.4100 - precision: 0.7991 - recall: 0.8565 - val_accuracy: 0.7978 - val_loss: 0.4094 - val_precision: 0.7600 - val_recall: 0.8636
Epoch 3/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 3s 57ms/step - accuracy: 0.8643 - loss: 0.3416 - precision: 0.8455 - recall: 0.8900 - val_accuracy: 0.7865 - val_loss: 0.3699 - val_precision: 0.7551 - val_recall: 0.8409
Epoch 4/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 3s 55ms/step - accuracy: 0.8667 - loss: 0.3061 - precision: 0.8430 - recall: 0.8995 - val_accuracy: 0.8090 - val_loss: 0.3446 - val_precision: 0.7647 - val_recall: 0.8864
Epoch 5/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - accuracy: 0.8786 - loss: 0.2809 - precision: 0.8657 - recall: 0.8947 - val_accuracy: 0.8427 - val_loss: 0.357

In [ ]:

# Evaluate the trained model on the held-out test dataset
test_results = model.evaluate(test_ds, return_dict=True, verbose=1)

print("\nBOVICARE TEST RESULTS")
for metric, value in test_results.items():
    print(f"{metric}: {value:.4f}")


6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 778ms/step - accuracy: 0.8913 - loss: 0.2763 - precision: 0.8750 - recall: 0.9130

BOVICARE TEST RESULTS
accuracy: 0.8913
loss: 0.2763
precision: 0.8750
recall: 0.9130


In [ ]:


from sklearn.metrics import confusion_matrix, classification_report

y_true = []
y_pred = []

for images, labels in test_ds:
    probabilities = model.predict(images, verbose=0).reshape(-1)

    y_true.extend(labels.numpy().reshape(-1).astype(int))
    y_pred.extend((probabilities >= 0.5).astype(int))

class_names = ["Mastitis", "Normal"]

print("Class order:", class_names)
print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=class_names,
    zero_division=0
))


Class order: ['Mastitis', 'Normal']

Confusion Matrix:
[[40  6]
 [ 4 42]]

Classification Report:
              precision    recall  f1-score   support

    Mastitis       0.91      0.87      0.89        46
      Normal       0.88      0.91      0.89        46

    accuracy                           0.89        92
   macro avg       0.89      0.89      0.89        92
weighted avg       0.89      0.89      0.89        92



In [ ]:

import os
from google.colab import files

# Save the trained model
model_path = "/content/bovicare_mastitis_model.keras"
model.save(model_path)

print("Model saved successfully!")
print("File size:", round(os.path.getsize(model_path) / (1024 * 1024), 2), "MB")

# Download the model to your computer
files.download(model_path)


Model saved successfully!
File size: 9.19 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import tensorflow as tf
from google.colab import files

# Convert the trained Keras model to TensorFlow Lite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the converted model
tflite_path = "/content/bovicare_mastitis_model.tflite"

with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print("TensorFlow Lite conversion successful!")
print("Model size:", round(len(tflite_model) / (1024 * 1024), 2), "MB")

# Download the mobile model
files.download(tflite_path)


Saved artifact at '/tmp/tmpic7rsexd'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_154')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  136720154325072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136720154325840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136720154325648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136720154326224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136720154319696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136720154326416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136720154325264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136720154326992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136720154325456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136720154324880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1367201543

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import os
import tensorflow as tf

tflite_path = "/content/bovicare_mastitis_model.tflite"

if os.path.exists(tflite_path):
    print("SUCCESS: TensorFlow Lite model exists.")
    print("File size:", round(os.path.getsize(tflite_path) / (1024 * 1024), 2), "MB")

    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()

    print("\nInput details:")
    print(interpreter.get_input_details())

    print("\nOutput details:")
    print(interpreter.get_output_details())
else:
    print("Model file not found. Run the conversion cell again.")


SUCCESS: TensorFlow Lite model exists.
File size: 8.48 MB

Input details:
[{'name': 'serving_default_keras_tensor_154:0', 'index': 0, 'shape': array([  1, 224, 224,   3], dtype=int32), 'shape_signature': array([ -1, 224, 224,   3], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0, 'block_size': 0}, 'sparsity_parameters': {}}]

Output details:
[{'name': 'StatefulPartitionedCall_1:0', 'index': 176, 'shape': array([1, 1], dtype=int32), 'shape_signature': array([-1,  1], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0, 'block_size': 0}, 'sparsity_parameters': {}}]


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:

from google.colab import files

files.download("/content/bovicare_mastitis_model.tflite")


FileNotFoundError: Cannot find file: /content/bovicare_mastitis_model.tflite

In [ ]:

from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [ ]:

import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if 'cow_milk_mastitis_dataset.csv' in files:
        print(os.path.join(root, 'cow_milk_mastitis_dataset.csv'))


In [ ]:

import os

drive_path = "/content/drive/MyDrive"

for item in os.listdir(drive_path):
    full_path = os.path.join(drive_path, item)
    if os.path.isdir(full_path):
        print("FOLDER:", item)
    else:
        print("FILE:", item)


FILE: my photo.mp4
FILE: मानक क्रमांक-11.pdf
FILE: मानक क्रमांक-46.pdf
FILE: मानक क्रमांक-123.pdf
FILE: मानक क्रमांक-117 (1).pdf
FILE: मानक क्रमांक-117.pdf
FILE: IMG_20250815_181545.jpg
FILE: IMG_20250815_232124.jpg
FOLDER: appsheet
FOLDER: Seva pustak
FILE: हिंद की चादर (1).pdf
FILE: हिंद की चादर.pdf
FILE: हिंद की चादर धामणगाव .pdf
FILE: हिंद की चादर धामणगाव .pptx
FILE: GPF_P13397_2024.pdf
FILE: Al use.pdf
FOLDER: Sqaff 2026
FOLDER: A SQAAF शाळा धामणगाव
FILE: .pdf.pdf
FOLDER: Google Earth
FOLDER: Classroom
FOLDER: Colab Notebooks
FOLDER: BoviCare_Dataset


In [ ]:

import os

dataset_path = "/content/drive/MyDrive/BoviCare_Dataset"

for root, dirs, files in os.walk(dataset_path):
    print("\nFolder:", root)
    for file in files:
        print("  File:", file)



Folder: /content/drive/MyDrive/BoviCare_Dataset
  File: archive (1).zip
  File: archive (2).zip
  File: indian-bovine-breed-classifier (1).ipynb
  File: archive (2) (1).zip


In [ ]:

import zipfile
import os

dataset_path = "/content/drive/MyDrive/BoviCare_Dataset"

for file in os.listdir(dataset_path):
    if file.lower().endswith(".zip"):
        zip_path = os.path.join(dataset_path, file)
        print(f"\nZIP: {file}")

        with zipfile.ZipFile(zip_path, "r") as z:
            matches = [
                name for name in z.namelist()
                if name.lower().endswith(".csv")
            ]
            print("CSV files:", matches if matches else "None found")



ZIP: archive (1).zip
CSV files: None found

ZIP: archive (2).zip
CSV files: ['cow_milk_mastitis_dataset.csv']

ZIP: archive (2) (1).zip
CSV files: None found


In [ ]:

import zipfile
import pandas as pd

zip_path = "/content/drive/MyDrive/BoviCare_Dataset/archive (2).zip"
csv_name = "cow_milk_mastitis_dataset.csv"

with zipfile.ZipFile(zip_path, "r") as z:
    with z.open(csv_name) as f:
        df = pd.read_csv(f)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
display(df.head())


Dataset loaded successfully!
Shape: (800, 9)

Columns:
['Cow_ID', 'Day', 'Milk_Temperature', 'Milk_pH', 'Milk_Conductivity', 'Somatic_Cell_Count', 'Milk_Yield', 'Clotting', 'class1']

First 5 rows:


,Cow_ID,Day,Milk_Temperature,Milk_pH,Milk_Conductivity,Somatic_Cell_Count,Milk_Yield,Clotting,class1
0,C0001,7,35.38,6.49,4.68,148,17.3,0,0
1,C0002,20,35.68,6.46,5.04,101,20.4,0,0
2,C0003,29,34.87,6.84,4.73,206,18.5,0,0
3,C0004,15,36.22,6.69,4.48,96,23.5,0,0
4,C0005,11,35.46,6.56,5.00,142,23.9,0,0


In [ ]:

print("All column names:")
print(df.columns.tolist())

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTarget column distribution:")
print(df["class1"].value_counts(dropna=False))

print("\nTarget proportions:")
print(df["class1"].value_counts(normalize=True, dropna=False).round(3))


All column names:
['Cow_ID', 'Day', 'Milk_Temperature', 'Milk_pH', 'Milk_Conductivity', 'Somatic_Cell_Count', 'Milk_Yield', 'Clotting', 'class1']

Missing values per column:
Cow_ID                0
Day                   0
Milk_Temperature      0
Milk_pH               0
Milk_Conductivity     0
Somatic_Cell_Count    0
Milk_Yield            0
Clotting              0
class1                0
dtype: int64

Duplicate rows: 0

Target column distribution:
class1
0    631
1    169
Name: count, dtype: int64

Target proportions:
class1
0    0.789
1    0.211
Name: proportion, dtype: float64


In [ ]:

print("Rows for class 0:")
display(df[df["class1"] == 0].head(3))

print("\nRows for class 1:")
display(df[df["class1"] == 1].head(3))

print("\nData types:")
print(df.dtypes)


Rows for class 0:


,Cow_ID,Day,Milk_Temperature,Milk_pH,Milk_Conductivity,Somatic_Cell_Count,Milk_Yield,Clotting,class1
0,C0001,7,35.38,6.49,4.68,148,17.3,0,0
1,C0002,20,35.68,6.46,5.04,101,20.4,0,0
2,C0003,29,34.87,6.84,4.73,206,18.5,0,0



Rows for class 1:


,Cow_ID,Day,Milk_Temperature,Milk_pH,Milk_Conductivity,Somatic_Cell_Count,Milk_Yield,Clotting,class1
7,C0008,21,38.48,6.89,7.33,733,6.9,0,1
11,C0012,23,38.37,6.93,6.29,762,10.8,1,1
14,C0015,24,37.88,7.03,6.47,597,5.5,0,1



Data types:
Cow_ID                 object
Day                     int64
Milk_Temperature      float64
Milk_pH               float64
Milk_Conductivity     float64
Somatic_Cell_Count      int64
Milk_Yield            float64
Clotting                int64
class1                  int64
dtype: object


In [ ]:

import zipfile

zip_path = "/content/drive/MyDrive/BoviCare_Dataset/archive (2).zip"

with zipfile.ZipFile(zip_path, "r") as z:
    for name in z.namelist():
        if not name.endswith("/"):
            print(name)


cow_milk_mastitis_dataset.csv


In [ ]:

# Compare average measurements for each target class
feature_cols = [
    "Milk_Temperature",
    "Milk_pH",
    "Milk_Conductivity",
    "Somatic_Cell_Count",
    "Milk_Yield",
    "Clotting"
]

print("Mean measurements by class:")
display(df.groupby("class1")[feature_cols].mean().round(2))

print("\nMinimum and maximum class values:")
print("class1:", sorted(df["class1"].dropna().unique().tolist()))


Mean measurements by class:


,Milk_Temperature,Milk_pH,Milk_Conductivity,Somatic_Cell_Count,Milk_Yield,Clotting
class1,,,,,,
0,35.54,6.65,4.69,147.99,20.15,0.00
1,38.05,7.11,6.78,598.05,9.35,0.82



Minimum and maximum class values:
class1: [0, 1]


In [ ]:

# Input features: measurable milk characteristics
feature_cols = [
    "Milk_Temperature",
    "Milk_pH",
    "Milk_Conductivity",
    "Somatic_Cell_Count",
    "Milk_Yield",
    "Clotting"
]

X = df[feature_cols].copy()
y = df["class1"].astype(int).copy()

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeatures used:")
print(X.columns.tolist())
print("\nTarget class counts:")
print(y.value_counts().sort_index())


Feature matrix shape: (800, 6)
Target shape: (800,)

Features used:
['Milk_Temperature', 'Milk_pH', 'Milk_Conductivity', 'Somatic_Cell_Count', 'Milk_Yield', 'Clotting']

Target class counts:
class1
0    631
1    169
Name: count, dtype: int64


In [ ]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts().sort_index())

print("\nTesting class distribution:")
print(y_test.value_counts().sort_index())


Training features: (640, 6)
Testing features: (160, 6)

Training class distribution:
class1
0    505
1    135
Name: count, dtype: int64

Testing class distribution:
class1
0    126
1     34
Name: count, dtype: int64


In [ ]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

rf_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    min_samples_leaf=2,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# Training-set score only; final evaluation comes later
print("Random Forest training completed!")
print("Training accuracy:", round(rf_model.score(X_train, y_train), 4))


Random Forest training completed!
Training accuracy: 1.0


In [ ]:

%pip install -q xgboost


In [ ]:

from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb_model.fit(X_train, y_train)

print("XGBoost training completed!")
print("Training accuracy:", round(xgb_model.score(X_train, y_train), 4))


XGBoost training completed!
Training accuracy: 1.0


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

models = {
    "Random Forest": rf_model,
    "XGBoost": xgb_model
}

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print(f"\n{'=' * 45}")
    print(name)
    print("=" * 45)
    print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
    print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 4))
    print("Recall:", round(recall_score(y_test, y_pred, zero_division=0), 4))
    print("F1-score:", round(f1_score(y_test, y_pred, zero_division=0), 4))
    print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))
    print("\nConfusion matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification report:")
    print(classification_report(y_test, y_pred, zero_division=0))



Random Forest
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1-score: 1.0
ROC-AUC: 1.0

Confusion matrix:
[[126   0]
 [  0  34]]

Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       126
           1       1.00      1.00      1.00        34

    accuracy                           1.00       160
   macro avg       1.00      1.00      1.00       160
weighted avg       1.00      1.00      1.00       160


XGBoost
Accuracy: 0.9875
Precision: 1.0
Recall: 0.9412
F1-score: 0.9697
ROC-AUC: 1.0

Confusion matrix:
[[126   0]
 [  2  32]]

Classification report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       126
           1       1.00      0.94      0.97        34

    accuracy                           0.99       160
   macro avg       0.99      0.97      0.98       160
weighted avg       0.99      0.99      0.99       160



In [ ]:

from sklearn.metrics import confusion_matrix

for name, model in {
    "Random Forest": rf_model,
    "XGBoost": xgb_model
}.items():
    predictions = model.predict(X_test)
    print(f"\n{name}")
    print(confusion_matrix(y_test, predictions))



Random Forest
[[126   0]
 [  0  34]]

XGBoost
[[126   0]
 [  2  32]]


In [ ]:

train_ids = set(df.loc[X_train.index, "Cow_ID"])
test_ids = set(df.loc[X_test.index, "Cow_ID"])

overlap = train_ids.intersection(test_ids)

print("Unique cows in training set:", len(train_ids))
print("Unique cows in test set:", len(test_ids))
print("Cows appearing in both sets:", len(overlap))

if overlap:
    print("Example overlapping IDs:", list(overlap)[:10])
else:
    print("No cow IDs overlap between training and test sets.")


Unique cows in training set: 640
Unique cows in test set: 160
Cows appearing in both sets: 0
No cow IDs overlap between training and test sets.


In [ ]:

# Check how strongly each feature is associated with the target
import pandas as pd

correlations = df[feature_cols + ["class1"]].corr(numeric_only=True)["class1"]
correlations = correlations.drop("class1").sort_values(
    key=abs, ascending=False
)

print("Feature correlations with class1:")
display(correlations.to_frame("correlation_with_class1").round(3))

print("\nTarget distribution by Clotting value:")
display(pd.crosstab(df["Clotting"], df["class1"], normalize="index").round(3))


Feature correlations with class1:


,correlation_with_class1
Somatic_Cell_Count,0.937
Milk_Conductivity,0.923
Milk_Temperature,0.901
Clotting,0.882
Milk_pH,0.881
Milk_Yield,-0.835



Target distribution by Clotting value:


class1,0,1
Clotting,,
0,0.953,0.047
1,0.000,1.000


In [ ]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

features_without_clotting = [
    col for col in feature_cols if col != "Clotting"
]

X_train_no_clotting = X_train[features_without_clotting]
X_test_no_clotting = X_test[features_without_clotting]

rf_no_clotting = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    min_samples_leaf=2,
    n_jobs=-1
)

rf_no_clotting.fit(X_train_no_clotting, y_train)

pred_no_clotting = rf_no_clotting.predict(X_test_no_clotting)

print("Accuracy without Clotting:",
      round(accuracy_score(y_test, pred_no_clotting), 4))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, pred_no_clotting))

print("\nClassification report:")
print(classification_report(y_test, pred_no_clotting, zero_division=0))


Accuracy without Clotting: 1.0

Confusion matrix:
[[126   0]
 [  0  34]]

Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       126
           1       1.00      1.00      1.00        34

    accuracy                           1.00       160
   macro avg       1.00      1.00      1.00       160
weighted avg       1.00      1.00      1.00       160



In [ ]:

# Compare feature ranges for each target class
for col in features_without_clotting:
    print(f"\n{col}")
    display(
        df.groupby("class1")[col]
        .agg(["min", "median", "max"])
        .round(2)
    )



Milk_Temperature


,min,median,max
class1,,,
0,34.04,35.53,37.10
1,36.89,38.01,39.38



Milk_pH


,min,median,max
class1,,,
0,6.35,6.65,7.04
1,6.85,7.11,7.42



Milk_Conductivity


,min,median,max
class1,,,
0,3.75,4.70,5.62
1,5.38,6.75,8.09



Somatic_Cell_Count


,min,median,max
class1,,,
0,34,148.0,274
1,235,591.0,1023



Milk_Yield


,min,median,max
class1,,,
0,8.9,20.0,28.6
1,4.7,9.4,15.0


In [ ]:

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Get class-1 probabilities from both models
rf_prob = rf_model.predict_proba(X_test)[:, 1]
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

# Equal-weight probability ensemble
hybrid_prob = (rf_prob + xgb_prob) / 2

# Use 0.5 as the initial decision threshold
hybrid_pred = (hybrid_prob >= 0.5).astype(int)

print("HYBRID MODEL: Random Forest + XGBoost")
print("Accuracy:", round(accuracy_score(y_test, hybrid_pred), 4))
print("Precision:", round(precision_score(y_test, hybrid_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, hybrid_pred, zero_division=0), 4))
print("F1-score:", round(f1_score(y_test, hybrid_pred, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, hybrid_prob), 4))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, hybrid_pred))


HYBRID MODEL: Random Forest + XGBoost
Accuracy: 0.9938
Precision: 1.0
Recall: 0.9706
F1-score: 0.9851
ROC-AUC: 1.0

Confusion matrix:
[[126   0]
 [  1  33]]


In [ ]:

from sklearn.base import clone
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Remove the potentially target-revealing feature
X_train_no_clotting = X_train.drop(columns=["Clotting"], errors="ignore")
X_test_no_clotting = X_test.drop(columns=["Clotting"], errors="ignore")

# Train fresh models without Clotting
rf_no_clotting = clone(rf_model)
xgb_no_clotting = clone(xgb_model)

rf_no_clotting.fit(X_train_no_clotting, y_train)
xgb_no_clotting.fit(X_train_no_clotting, y_train)

# Combine probabilities
rf_prob_nc = rf_no_clotting.predict_proba(X_test_no_clotting)[:, 1]
xgb_prob_nc = xgb_no_clotting.predict_proba(X_test_no_clotting)[:, 1]

hybrid_prob_nc = (rf_prob_nc + xgb_prob_nc) / 2
hybrid_pred_nc = (hybrid_prob_nc >= 0.5).astype(int)

print("HYBRID MODEL WITHOUT CLOTTING")
print("Accuracy:", round(accuracy_score(y_test, hybrid_pred_nc), 4))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, hybrid_pred_nc))
print("\nClassification report:")
print(classification_report(y_test, hybrid_pred_nc, zero_division=0))


HYBRID MODEL WITHOUT CLOTTING
Accuracy: 0.9938

Confusion matrix:
[[126   0]
 [  1  33]]

Classification report:
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       126
           1       1.00      0.97      0.99        34

    accuracy                           0.99       160
   macro avg       1.00      0.99      0.99       160
weighted avg       0.99      0.99      0.99       160



In [ ]:

# Check how strongly each feature is associated with the target
feature_cols = [
    "Milk_Temperature",
    "Milk_pH",
    "Milk_Conductivity",
    "Somatic_Cell_Count",
    "Milk_Yield"
]

correlations = (
    df[feature_cols + ["class1"]]
    .corr(numeric_only=True)["class1"]
    .drop("class1")
    .sort_values(key=abs, ascending=False)
)

print("Feature correlations with class1:")
display(correlations.to_frame("Correlation"))

# Inspect the target values for each feature's distinct values
print("\nNumber of distinct values per feature:")
print(df[feature_cols].nunique().sort_values())


Feature correlations with class1:


,Correlation
Somatic_Cell_Count,0.936517
Milk_Conductivity,0.923199
Milk_Temperature,0.901365
Milk_pH,0.881047
Milk_Yield,-0.835462



Number of distinct values per feature:
Milk_pH                95
Milk_Yield            205
Milk_Conductivity     252
Milk_Temperature      311
Somatic_Cell_Count    320
dtype: int64


In [ ]:

# Check whether identical measurements appear with different labels
feature_cols = [
    "Milk_Temperature",
    "Milk_pH",
    "Milk_Conductivity",
    "Somatic_Cell_Count",
    "Milk_Yield"
]

duplicate_features = df[
    df.duplicated(subset=feature_cols, keep=False)
].sort_values(feature_cols)

print("Rows sharing identical feature values:", len(duplicate_features))

if len(duplicate_features):
    print("\nExamples of repeated feature patterns:")
    display(duplicate_features[
        feature_cols + ["class1"]
    ].head(20))

    label_counts = (
        duplicate_features.groupby(feature_cols)["class1"]
        .nunique()
    )
    print(
        "\nRepeated feature patterns with conflicting labels:",
        int((label_counts > 1).sum())
    )
else:
    print("No identical feature rows found.")


Rows sharing identical feature values: 0
No identical feature rows found.


In [ ]:

import pandas as pd

feature_cols = [
    "Milk_Temperature",
    "Milk_pH",
    "Milk_Conductivity",
    "Somatic_Cell_Count",
    "Milk_Yield"
]

for col in feature_cols:
    print(f"\n--- {col} by class1 ---")
    display(
        df.groupby("class1")[col]
        .agg(["min", "median", "max", "mean"])
        .round(2)
    )



--- Milk_Temperature by class1 ---


,min,median,max,mean
class1,,,,
0,34.04,35.53,37.10,35.54
1,36.89,38.01,39.38,38.05



--- Milk_pH by class1 ---


,min,median,max,mean
class1,,,,
0,6.35,6.65,7.04,6.65
1,6.85,7.11,7.42,7.11



--- Milk_Conductivity by class1 ---


,min,median,max,mean
class1,,,,
0,3.75,4.70,5.62,4.69
1,5.38,6.75,8.09,6.78



--- Somatic_Cell_Count by class1 ---


,min,median,max,mean
class1,,,,
0,34,148.0,274,147.99
1,235,591.0,1023,598.05



--- Milk_Yield by class1 ---


,min,median,max,mean
class1,,,,
0,8.9,20.0,28.6,20.15
1,4.7,9.4,15.0,9.35


In [ ]:

# Check whether labels are strongly associated with cow IDs or day values

print("Unique Cow_ID values:", df["Cow_ID"].nunique())
print("Unique Day values:", df["Day"].nunique())

print("\nClass distribution by Day:")
display(pd.crosstab(df["Day"], df["class1"], normalize="index").round(3).head(15))

print("\nNumber of Cow_IDs associated with each class:")
print(df.groupby("Cow_ID")["class1"].nunique().value_counts().sort_index())

print("\nCow_IDs associated with both classes:")
mixed_cows = (
    df.groupby("Cow_ID")["class1"]
    .nunique()
    .loc[lambda s: s > 1]
)
print(mixed_cows.index.tolist()[:20])
print("Total:", len(mixed_cows))


Unique Cow_ID values: 800
Unique Day values: 30

Class distribution by Day:


class1,0,1
Day,,
1,0.833,0.167
2,0.792,0.208
3,0.861,0.139
4,0.560,0.440
5,0.690,0.310
6,0.786,0.214
7,0.931,0.069
8,0.800,0.200
9,0.783,0.217



Number of Cow_IDs associated with each class:
class1
1    800
Name: count, dtype: int64

Cow_IDs associated with both classes:
[]
Total: 0


In [ ]:

# Check whether class labels are associated with the order of Cow_IDs

df_check = df.copy()

df_check["Cow_Number"] = (
    df_check["Cow_ID"].astype(str)
    .str.extract(r"(\d+)", expand=False)
    .astype(int)
)

print("Cow number range:")
print(df_check["Cow_Number"].min(), "to", df_check["Cow_Number"].max())

print("\nClass distribution by Cow_ID number quartile:")
df_check["Cow_Group"] = pd.qcut(
    df_check["Cow_Number"],
    q=4,
    duplicates="drop"
)

display(
    pd.crosstab(
        df_check["Cow_Group"],
        df_check["class1"],
        normalize="index"
    ).round(3)
)

print("\nFirst 20 rows ordered by Cow_ID:")
display(
    df_check.sort_values("Cow_Number")[
        ["Cow_ID", "Day", "class1"]
    ].head(20)
)


Cow number range:
1 to 800

Class distribution by Cow_ID number quartile:


class1,0,1
Cow_Group,,
"(0.999, 200.75]",0.765,0.235
"(200.75, 400.5]",0.805,0.195
"(400.5, 600.25]",0.810,0.190
"(600.25, 800.0]",0.775,0.225



First 20 rows ordered by Cow_ID:


,Cow_ID,Day,class1
0,C0001,7,0
1,C0002,20,0
2,C0003,29,0
3,C0004,15,0
4,C0005,11,0
5,C0006,8,0
6,C0007,29,0
7,C0008,21,1
8,C0009,7,0
9,C0010,26,0


In [ ]:

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import pandas as pd

feature_cols = [
    "Milk_Temperature",
    "Milk_pH",
    "Milk_Conductivity",
    "Somatic_Cell_Count",
    "Milk_Yield"
]

X_cv = df[feature_cols].copy()
y_cv = df["class1"].astype(int)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        min_samples_leaf=2,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    )
}

scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

for name, model in models.items():
    scores = cross_validate(
        model, X_cv, y_cv,
        cv=cv,
        scoring=scoring,
        n_jobs=1
    )

    print(f"\n{name} — 5-fold cross-validation")
    for metric in scoring:
        values = scores[f"test_{metric}"]
        print(
            f"{metric}: {values.mean():.4f} "
            f"+/- {values.std():.4f}"
        )



Random Forest — 5-fold cross-validation
accuracy: 1.0000 +/- 0.0000
precision: 1.0000 +/- 0.0000
recall: 1.0000 +/- 0.0000
f1: 1.0000 +/- 0.0000
roc_auc: 1.0000 +/- 0.0000

XGBoost — 5-fold cross-validation
accuracy: 0.9975 +/- 0.0031
precision: 0.9941 +/- 0.0118
recall: 0.9941 +/- 0.0118
f1: 0.9940 +/- 0.0073
roc_auc: 1.0000 +/- 0.0000


In [ ]:

# Check whether individual features separate the two classes unusually well

feature_cols = [
    "Milk_Temperature",
    "Milk_pH",
    "Milk_Conductivity",
    "Somatic_Cell_Count",
    "Milk_Yield"
]

for col in feature_cols:
    print(f"\n--- {col} ---")

    grouped = df.groupby("class1")[col].agg(
        ["min", "max", "mean", "median"]
    )
    display(grouped.round(3))

    print("Overlapping range:")
    low = max(
        df.loc[df["class1"] == 0, col].min(),
        df.loc[df["class1"] == 1, col].min()
    )
    high = min(
        df.loc[df["class1"] == 0, col].max(),
        df.loc[df["class1"] == 1, col].max()
    )
    print("Yes" if low <= high else "No")



--- Milk_Temperature ---


,min,max,mean,median
class1,,,,
0,34.04,37.10,35.541,35.53
1,36.89,39.38,38.051,38.01


Overlapping range:
Yes

--- Milk_pH ---


,min,max,mean,median
class1,,,,
0,6.35,7.04,6.647,6.65
1,6.85,7.42,7.110,7.11


Overlapping range:
Yes

--- Milk_Conductivity ---


...

Overlapping range:
Yes

--- Somatic_Cell_Count ---


,min,max,mean,median
class1,,,,
0,34,274,147.994,148.0
1,235,1023,598.053,591.0


Overlapping range:
Yes

--- Milk_Yield ---


,min,max,mean,median
class1,,,,
0,8.9,28.6,20.148,20.0
1,4.7,15.0,9.354,9.4


Overlapping range:
Yes


In [ ]:

from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier

feature_cols = [
    "Milk_Temperature",
    "Milk_pH",
    "Milk_Conductivity",
    "Somatic_Cell_Count",
    "Milk_Yield"
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Single-feature cross-validation accuracy:\n")

for col in feature_cols:
    model = DecisionTreeClassifier(
        max_depth=2,
        random_state=42
    )

    scores = cross_val_score(
        model,
        df[[col]],
        df["class1"].astype(int),
        cv=cv,
        scoring="accuracy"
    )

    print(
        f"{col}: {scores.mean():.4f} "
        f"+/- {scores.std():.4f}"
    )


Single-feature cross-validation accuracy:

Milk_Temperature: 0.9988 +/- 0.0025
Milk_pH: 0.9938 +/- 0.0068
Milk_Conductivity: 0.9988 +/- 0.0025
Somatic_Cell_Count: 0.9963 +/- 0.0050
Milk_Yield: 0.9850 +/- 0.0094


In [ ]:

# Inspect the dataset for clues about its source and construction

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 10 records:")
display(df.head(10))

print("\nLast 10 records:")
display(df.tail(10))

print("\nTarget distribution:")
print(df["class1"].value_counts().sort_index())

print("\nClass proportions:")
print(df["class1"].value_counts(normalize=True).sort_index().round(4))


Dataset shape: (800, 9)

Column names:
['Cow_ID', 'Day', 'Milk_Temperature', 'Milk_pH', 'Milk_Conductivity', 'Somatic_Cell_Count', 'Milk_Yield', 'Clotting', 'class1']

First 10 records:


,Cow_ID,Day,Milk_Temperature,Milk_pH,Milk_Conductivity,Somatic_Cell_Count,Milk_Yield,Clotting,class1
0,C0001,7,35.38,6.49,4.68,148,17.3,0,0
1,C0002,20,35.68,6.46,5.04,101,20.4,0,0
2,C0003,29,34.87,6.84,4.73,206,18.5,0,0
3,C0004,15,36.22,6.69,4.48,96,23.5,0,0
4,C0005,11,35.46,6.56,5.00,142,23.9,0,0
5,C0006,8,36.06,6.70,4.63,108,22.6,0,0
6,C0007,29,35.67,6.39,4.86,173,20.9,0,0
7,C0008,21,38.48,6.89,7.33,733,6.9,0,1
8,C0009,7,35.78,6.68,4.51,129,16.4,0,0
9,C0010,26,35.72,6.74,4.73,184,25.8,0,0



Last 10 records:


,Cow_ID,Day,Milk_Temperature,Milk_pH,Milk_Conductivity,Somatic_Cell_Count,Milk_Yield,Clotting,class1
790,C0791,14,37.65,7.16,6.97,528,11.8,1,1
791,C0792,26,35.42,6.63,4.64,43,17.8,0,0
792,C0793,5,35.54,6.55,4.72,80,15.7,0,0
793,C0794,3,37.85,7.13,6.07,491,9.7,1,1
794,C0795,23,35.04,6.55,5.22,153,23.3,0,0
795,C0796,29,35.62,6.81,5.01,135,19.9,0,0
796,C0797,11,35.99,6.62,4.48,87,21.9,0,0
797,C0798,11,35.75,6.81,4.30,117,25.0,0,0
798,C0799,18,38.51,7.05,6.79,492,9.7,1,1
799,C0800,15,36.00,6.64,4.83,173,18.3,0,0



Target distribution:
class1
0    631
1    169
Name: count, dtype: int64

Class proportions:
class1
0    0.7887
1    0.2112
Name: proportion, dtype: float64


In [ ]:

import zipfile

zip_path = "/content/drive/MyDrive/BoviCare_Dataset/archive (2).zip"

with zipfile.ZipFile(zip_path, "r") as z:
    print("Files inside archive (2).zip:")
    for name in z.namelist():
        if not name.endswith("/"):
            print(name)


Files inside archive (2).zip:
cow_milk_mastitis_dataset.csv


In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/BoviCare_Dataset/archive (2).zip"

with zipfile.ZipFile(zip_path, "r") as z:
    with z.open("cow_milk_mastitis_dataset.csv") as f:
        print(f.read(1500).decode("utf-8", errors="replace"))

Cow_ID,Day,Milk_Temperature,Milk_pH,Milk_Conductivity,Somatic_Cell_Count,Milk_Yield,Clotting,class1
C0001,7,35.38,6.49,4.68,148,17.3,0,0
C0002,20,35.68,6.46,5.04,101,20.4,0,0
C0003,29,34.87,6.84,4.73,206,18.5,0,0
C0004,15,36.22,6.69,4.48,96,23.5,0,0
C0005,11,35.46,6.56,5.0,142,23.9,0,0
C0006,8,36.06,6.7,4.63,108,22.6,0,0
C0007,29,35.67,6.39,4.86,173,20.9,0,0
C0008,21,38.48,6.89,7.33,733,6.9,0,1
C0009,7,35.78,6.68,4.51,129,16.4,0,0
C0010,26,35.72,6.74,4.73,184,25.8,0,0
C0011,19,35.82,6.55,4.7,151,17.7,0,0
C0012,23,38.37,6.93,6.29,762,10.8,1,1
C0013,11,35.6,6.56,4.62,111,23.2,0,0
C0014,11,35.85,6.68,4.15,94,18.3,0,0
C0015,24,37.88,7.03,6.47,597,5.5,0,1
C0016,21,37.58,7.3,6.68,685,15.0,1,1
C0017,4,39.09,7.05,6.33,777,12.5,1,1
C0018,8,36.4,6.72,4.63,147,18.6,0,0
C0019,24,35.48,6.74,5.0,158,16.1,0,0
C0020,3,34.78,6.63,4.49,85,18.5,0,0
C0021,22,35.56,6.65,4.24,226,20.6,0,0
C0022,21,35.16,6.51,4.65,143,20.2,0,0
C0023,2,35.92,6.62,4.55,178,18.8,0,0
C0024,24,38.1,7.28,5.76,544,7.4,1,1
C0025,12,

In [ ]:
import pandas as pd
import zipfile

zip_path = "/content/drive/MyDrive/BoviCare_Dataset/archive (2).zip"

with zipfile.ZipFile(zip_path, "r") as z:
    with z.open("cow_milk_mastitis_dataset.csv") as f:
        df_check = pd.read_csv(f)

print("Dataset shape:", df_check.shape)
print("\nColumn names:")
print(df_check.columns.tolist())

print("\nFirst 5 rows:")
display(df_check.head())

print("\nUnique values in target column:")
print(df_check["class1"].value_counts(dropna=False))

print("\nDataset statistical summary:")
display(df_check.describe().round(2))

Dataset shape: (800, 9)

Column names:
['Cow_ID', 'Day', 'Milk_Temperature', 'Milk_pH', 'Milk_Conductivity', 'Somatic_Cell_Count', 'Milk_Yield', 'Clotting', 'class1']

First 5 rows:


,Cow_ID,Day,Milk_Temperature,Milk_pH,Milk_Conductivity,Somatic_Cell_Count,Milk_Yield,Clotting,class1
0,C0001,7,35.38,6.49,4.68,148,17.3,0,0
1,C0002,20,35.68,6.46,5.04,101,20.4,0,0
2,C0003,29,34.87,6.84,4.73,206,18.5,0,0
3,C0004,15,36.22,6.69,4.48,96,23.5,0,0
4,C0005,11,35.46,6.56,5.00,142,23.9,0,0



Unique values in target column:
class1
0    631
1    169
Name: count, dtype: int64

Dataset statistical summary:


,Day,Milk_Temperature,Milk_pH,Milk_Conductivity,Somatic_Cell_Count,Milk_Yield,Clotting,class1
count,800.00,800.00,800.00,800.00,800.00,800.00,800.00,800.00
mean,16.00,36.07,6.74,5.13,243.07,17.87,0.17,0.21
std,9.09,1.14,0.21,0.93,196.29,5.28,0.38,0.41
min,1.00,34.04,6.35,3.75,34.00,4.70,0.00,0.00
25%,8.00,35.32,6.60,4.55,127.00,15.30,0.00,0.00
50%,16.50,35.72,6.68,4.81,163.00,18.90,0.00,0.00
75%,24.00,36.36,6.82,5.13,213.00,21.62,0.00,0.00
max,30.00,39.38,7.42,8.09,1023.00,28.60,1.00,1.00


In [ ]:
import pandas as pd

# Compare measurements for the two classes
features = [
    "Milk_Temperature",
    "Milk_pH",
    "Milk_Conductivity",
    "Somatic_Cell_Count",
    "Milk_Yield"
]

print("Mean measurements for each class:")
display(df.groupby("class1")[features].mean().round(3).T)

print("\nClotting values by class:")
display(pd.crosstab(df["class1"], df["Clotting"], normalize="index").round(3))

print("\nLabels by Day:")
display(pd.crosstab(df["Day"], df["class1"]).head(10))

Mean measurements for each class:


class1,0,1
Milk_Temperature,35.541,38.051
Milk_pH,6.647,7.110
Milk_Conductivity,4.689,6.780
Somatic_Cell_Count,147.994,598.053
Milk_Yield,20.148,9.354



Clotting values by class:


Clotting,0,1
class1,,
0,1.000,0.000
1,0.183,0.817



Labels by Day:


class1,0,1
Day,,
1,30,6
2,19,5
3,31,5
4,14,11
5,20,9
6,11,3
7,27,2
8,20,5
9,18,5


In [ ]:
# Inspect all days and their class distributions
day_summary = pd.crosstab(df["Day"], df["class1"])
day_summary["Total"] = day_summary.sum(axis=1)

print("Records and labels across all days:")
display(day_summary)

print("\nFirst 15 records, sorted by Cow_ID:")
display(df.sort_values("Cow_ID").head(15))

print("\nNumber of unique cows:", df["Cow_ID"].nunique())
print("Number of duplicate complete rows:", df.duplicated().sum())

Records and labels across all days:


class1,0,1,Total
Day,,,
1,30,6,36
2,19,5,24
3,31,5,36
4,14,11,25
5,20,9,29
6,11,3,14
7,27,2,29
8,20,5,25
9,18,5,23



First 15 records, sorted by Cow_ID:


,Cow_ID,Day,Milk_Temperature,Milk_pH,Milk_Conductivity,Somatic_Cell_Count,Milk_Yield,Clotting,class1
0,C0001,7,35.38,6.49,4.68,148,17.3,0,0
1,C0002,20,35.68,6.46,5.04,101,20.4,0,0
2,C0003,29,34.87,6.84,4.73,206,18.5,0,0
3,C0004,15,36.22,6.69,4.48,96,23.5,0,0
4,C0005,11,35.46,6.56,5.00,142,23.9,0,0
5,C0006,8,36.06,6.70,4.63,108,22.6,0,0
6,C0007,29,35.67,6.39,4.86,173,20.9,0,0
7,C0008,21,38.48,6.89,7.33,733,6.9,0,1
8,C0009,7,35.78,6.68,4.51,129,16.4,0,0
9,C0010,26,35.72,6.74,4.73,184,25.8,0,0



Number of unique cows: 800
Number of duplicate complete rows: 0


In [ ]:
print("Class distribution across all 30 days:")
display(day_summary)

print("\nPercentage of Class 1 for each day:")
display(
    (df.groupby("Day")["class1"].mean() * 100)
    .round(1)
    .rename("Class_1_Percentage")
    .to_frame()
)

Class distribution across all 30 days:


class1,0,1,Total
Day,,,
1,30,6,36
2,19,5,24
3,31,5,36
4,14,11,25
5,20,9,29
6,11,3,14
7,27,2,29
8,20,5,25
9,18,5,23



Percentage of Class 1 for each day:


,Class_1_Percentage
Day,
1,16.7
2,20.8
3,13.9
4,44.0
5,31.0
6,21.4
7,6.9
8,20.0
9,21.7


In [ ]:
import pandas as pd

# Compare each measurement with the target label
features = [
    "Milk_Temperature",
    "Milk_pH",
    "Milk_Conductivity",
    "Somatic_Cell_Count",
    "Milk_Yield",
    "Clotting"
]

for feature in features:
    print(f"\n--- {feature} ---")
    display(
        df.groupby("class1")[feature]
        .agg(["min", "median", "mean", "max"])
        .round(2)
    )

# Show a few records from both classes
print("\nSample records where class1 = 0:")
display(df[df["class1"] == 0].head(5))

print("\nSample records where class1 = 1:")
display(df[df["class1"] == 1].head(5))


--- Milk_Temperature ---


,min,median,mean,max
class1,,,,
0,34.04,35.53,35.54,37.10
1,36.89,38.01,38.05,39.38



--- Milk_pH ---


,min,median,mean,max
class1,,,,
0,6.35,6.65,6.65,7.04
1,6.85,7.11,7.11,7.42



--- Milk_Conductivity ---


...

,min,median,mean,max
class1,,,,
0,0,0.0,0.00,0
1,0,1.0,0.82,1



Sample records where class1 = 0:


,Cow_ID,Day,Milk_Temperature,Milk_pH,Milk_Conductivity,Somatic_Cell_Count,Milk_Yield,Clotting,class1
0,C0001,7,35.38,6.49,4.68,148,17.3,0,0
1,C0002,20,35.68,6.46,5.04,101,20.4,0,0
2,C0003,29,34.87,6.84,4.73,206,18.5,0,0
3,C0004,15,36.22,6.69,4.48,96,23.5,0,0
4,C0005,11,35.46,6.56,5.00,142,23.9,0,0



Sample records where class1 = 1:


,Cow_ID,Day,Milk_Temperature,Milk_pH,Milk_Conductivity,Somatic_Cell_Count,Milk_Yield,Clotting,class1
7,C0008,21,38.48,6.89,7.33,733,6.9,0,1
11,C0012,23,38.37,6.93,6.29,762,10.8,1,1
14,C0015,24,37.88,7.03,6.47,597,5.5,0,1
15,C0016,21,37.58,7.30,6.68,685,15.0,1,1
16,C0017,4,39.09,7.05,6.33,777,12.5,1,1


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

features_with_clotting = [
    "Milk_Temperature",
    "Milk_pH",
    "Milk_Conductivity",
    "Somatic_Cell_Count",
    "Milk_Yield",
    "Clotting"
]

features_without_clotting = [
    "Milk_Temperature",
    "Milk_pH",
    "Milk_Conductivity",
    "Somatic_Cell_Count",
    "Milk_Yield"
]

X_with = df[features_with_clotting]
X_without = df[features_without_clotting]
y = df["class1"]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, X in [
    ("With Clotting", X_with),
    ("Without Clotting", X_without)
]:
    model = DecisionTreeClassifier(max_depth=2, random_state=42)
    scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")

    print(f"{name}:")
    print("Fold accuracies:", scores.round(4))
    print("Mean accuracy:", round(scores.mean(), 4))
    print()

With Clotting:
Fold accuracies: [0.9938 0.9875 1.     1.     1.    ]
Mean accuracy: 0.9962

Without Clotting:
Fold accuracies: [0.9938 0.9875 1.     1.     1.    ]
Mean accuracy: 0.9962



In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = [
    "Milk_Temperature",
    "Milk_pH",
    "Milk_Conductivity",
    "Somatic_Cell_Count",
    "Milk_Yield"
]

X = df[features]
y = df["class1"]

tree = DecisionTreeClassifier(max_depth=2, random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- Milk_Temperature <= 36.84
|   |--- class: 0
|--- Milk_Temperature >  36.84
|   |--- Milk_pH <= 6.84
|   |   |--- class: 0
|   |--- Milk_pH >  6.84
|   |   |--- class: 1



In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report

features = [
    "Milk_Temperature",
    "Milk_pH",
    "Milk_Conductivity",
    "Somatic_Cell_Count",
    "Milk_Yield"
]

X = df[features]
y = df["class1"]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

tree = DecisionTreeClassifier(max_depth=2, random_state=42)
predictions = cross_val_predict(tree, X, y, cv=cv)

print("Confusion matrix:")
print(confusion_matrix(y, predictions))

print("\nClassification report:")
print(classification_report(y, predictions, digits=4))

mistakes = df.loc[predictions != y].copy()
mistakes["Predicted_class"] = predictions[predictions != y]

print("\nNumber of misclassified records:", len(mistakes))
display(mistakes)

Confusion matrix:
[[629   2]
 [  1 168]]

Classification report:
              precision    recall  f1-score   support

           0     0.9984    0.9968    0.9976       631
           1     0.9882    0.9941    0.9912       169

    accuracy                         0.9962       800
   macro avg     0.9933    0.9955    0.9944       800
weighted avg     0.9963    0.9962    0.9963       800


Number of misclassified records: 3


,Cow_ID,Day,Milk_Temperature,Milk_pH,Milk_Conductivity,Somatic_Cell_Count,Milk_Yield,Clotting,class1,Predicted_class
46,C0047,30,36.40,6.59,5.03,274,24.2,0,0,1
665,C0666,24,37.10,6.83,4.63,123,24.0,0,0,1
755,C0756,17,37.59,7.12,5.76,235,7.5,1,1,0


In [ ]:
cols = [
    "Cow_ID",
    "Day",
    "Milk_Temperature",
    "Milk_pH",
    "Milk_Conductivity",
    "Somatic_Cell_Count",
    "Milk_Yield",
    "Clotting",
    "class1",
    "Predicted_class"
]

display(mistakes[cols].sort_values("Cow_ID"))

,Cow_ID,Day,Milk_Temperature,Milk_pH,Milk_Conductivity,Somatic_Cell_Count,Milk_Yield,Clotting,class1,Predicted_class
46,C0047,30,36.40,6.59,5.03,274,24.2,0,0,1
665,C0666,24,37.10,6.83,4.63,123,24.0,0,0,1
755,C0756,17,37.59,7.12,5.76,235,7.5,1,1,0


In [ ]:
print(
    mistakes[
        [
            "Cow_ID",
            "Milk_Temperature",
            "Milk_pH",
            "Milk_Conductivity",
            "Somatic_Cell_Count",
            "Milk_Yield",
            "Clotting",
            "class1",
            "Predicted_class"
        ]
    ].to_string(index=False)
)

Cow_ID  Milk_Temperature  Milk_pH  Milk_Conductivity  Somatic_Cell_Count  Milk_Yield  Clotting  class1  Predicted_class
 C0047             36.40     6.59               5.03                 274        24.2         0       0                1
 C0666             37.10     6.83               4.63                 123        24.0         0       0                1
 C0756             37.59     7.12               5.76                 235         7.5         1       1                0


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import numpy as np

features = [
    "Milk_Temperature",
    "Milk_pH",
    "Milk_Conductivity",
    "Somatic_Cell_Count",
    "Milk_Yield"
]

X = df[features]
y = df["class1"]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
}

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

for name, model in models.items():
    results = cross_validate(
        model, X, y,
        cv=cv,
        scoring=scoring,
        n_jobs=1
    )

    print(f"\n{name}")
    for metric in scoring:
        scores = results[f"test_{metric}"]
        print(
            f"{metric}: {scores.mean():.4f} "
            f"+/- {scores.std():.4f}"
        )


Random Forest
accuracy: 1.0000 +/- 0.0000
precision: 1.0000 +/- 0.0000
recall: 1.0000 +/- 0.0000
f1: 1.0000 +/- 0.0000
roc_auc: 1.0000 +/- 0.0000

XGBoost
accuracy: 0.9975 +/- 0.0031
precision: 0.9941 +/- 0.0118
recall: 0.9941 +/- 0.0118
f1: 0.9940 +/- 0.0073
roc_auc: 1.0000 +/- 0.0000


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

features = [
    "Milk_Temperature",
    "Milk_pH",
    "Milk_Conductivity",
    "Somatic_Cell_Count",
    "Milk_Yield"
]

X = df[features]
y = df["class1"]

rf = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X, y)

importance = pd.DataFrame({
    "Feature": features,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)

display(importance)

print("Importance total:", importance["Importance"].sum())

,Feature,Importance
0,Milk_Temperature,0.337565
2,Milk_Conductivity,0.290126
3,Somatic_Cell_Count,0.232651
1,Milk_pH,0.115542
4,Milk_Yield,0.024116


Importance total: 1.0


In [ ]:
display(importance)

print("Importance total:", importance["Importance"].sum())

,Feature,Importance
0,Milk_Temperature,0.337565
2,Milk_Conductivity,0.290126
3,Somatic_Cell_Count,0.232651
1,Milk_pH,0.115542
4,Milk_Yield,0.024116


Importance total: 1.0


In [ ]:
import joblib
import os

output_dir = "/content/drive/MyDrive/BoviCare_Model"
os.makedirs(output_dir, exist_ok=True)

model_path = os.path.join(output_dir, "bovicare_mastitis_rf.pkl")

joblib.dump(
    {
        "model": rf,
        "features": features,
        "target_column": "class1",
        "class_meanings": {
            "0": "Unverified class 0",
            "1": "Unverified class 1"
        }
    },
    model_path
)

print("Model saved to:", model_path)
print("File exists:", os.path.exists(model_path))

Model saved to: /content/drive/MyDrive/BoviCare_Model/bovicare_mastitis_rf.pkl
File exists: True


In [ ]:
import joblib
import pandas as pd

model_path = "/content/drive/MyDrive/BoviCare_Model/bovicare_mastitis_rf.pkl"

saved_data = joblib.load(model_path)
loaded_model = saved_data["model"]
loaded_features = saved_data["features"]

# Test with one record from the dataset
sample = df[loaded_features].iloc[[0]]
prediction = loaded_model.predict(sample)[0]
probabilities = loaded_model.predict_proba(sample)[0]

print("Input measurements:")
display(sample)

print("Predicted class:", prediction)
print("Class probabilities:", probabilities)
print("Loaded features:", loaded_features)

Input measurements:


,Milk_Temperature,Milk_pH,Milk_Conductivity,Somatic_Cell_Count,Milk_Yield
0,35.38,6.49,4.68,148,17.3


Predicted class: 0
Class probabilities: [1. 0.]
Loaded features: ['Milk_Temperature', 'Milk_pH', 'Milk_Conductivity', 'Somatic_Cell_Count', 'Milk_Yield']


In [ ]:
import joblib
import pandas as pd

model_path = "/content/drive/MyDrive/BoviCare_Model/bovicare_mastitis_rf.pkl"

saved_data = joblib.load(model_path)
model = saved_data["model"]
features = saved_data["features"]

# Select five records
test_rows = df[features].iloc[:5]

predictions = model.predict(test_rows)
probabilities = model.predict_proba(test_rows)

results = test_rows.copy()
results["Actual_class"] = df["class1"].iloc[:5].values
results["Predicted_class"] = predictions
results["Class_0_score"] = probabilities[:, 0]
results["Class_1_score"] = probabilities[:, 1]

display(results.round(3))

,Milk_Temperature,Milk_pH,Milk_Conductivity,Somatic_Cell_Count,Milk_Yield,Actual_class,Predicted_class,Class_0_score,Class_1_score
0,35.38,6.49,4.68,148,17.3,0,0,1.0,0.0
1,35.68,6.46,5.04,101,20.4,0,0,1.0,0.0
2,34.87,6.84,4.73,206,18.5,0,0,1.0,0.0
3,36.22,6.69,4.48,96,23.5,0,0,1.0,0.0
4,35.46,6.56,5.00,142,23.9,0,0,1.0,0.0


In [ ]:
print("Actual labels:   ", df["class1"].iloc[:5].tolist())
print("Predicted labels:", predictions.tolist())

print("\nClass probabilities:")
print(probabilities.round(3))

Actual labels:    [0, 0, 0, 0, 0]
Predicted labels: [0, 0, 0, 0, 0]

Class probabilities:
[[1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]]


In [ ]:
from google.colab import files

model_path = "/content/drive/MyDrive/BoviCare_Model/bovicare_mastitis_rf.pkl"

files.download(model_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>